In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
#import cmocean
import cartopy
#cartopy.config['pre_existing_data_dir'] = '/home1/datawork/kbalem/cartopy_shapefiles/'
#cartopy.config['data_dir'] = '/home1/datawork/kbalem/cartopy_shapefiles/'
#cartopy.config['repo_data_dir'] = '/home1/datawork/kbalem/cartopy_shapefiles/'
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.geodesic import Geodesic
Geodesic = Geodesic()
land_color = [85/255, 92/255, 105/255]
land_feature=cfeature.NaturalEarthFeature(category='physical',name='land',scale='50m',facecolor=land_color)
plt.rcParams['axes.grid'] = True
import numpy as np
import xarray as xr
import pandas as pd

import scipy.stats as stats
from sklearn.metrics import mean_squared_error
from math import radians
from sklearn.metrics.pairwise import haversine_distances

In [ ]:
#groundings_arvor = xr.open_dataset('/home1/datawork/kbalem/DeepArvorGrounding_ABYSS_June24.nc')
#groundings_arvor = groundings_arvor[['lat_gps','lon_gps','grnd','wmo','cycle']].to_dataframe()
#groundings_arvor.columns = ['lat','lon','grnd','wmo','cycle']
groundings_arvor = pd.read_csv('Deep_Arvor_dos_paper2_21Jan2025_2.txt',sep=' ',names=['WMO', 'lon', 'lat', 'grnd', 'uncertainty', 'tid_flag', 'SID','gebco'])
groundings_arvor.head()

In [ ]:
f=plt.figure()
ax=f.add_subplot(projection=ccrs.Mercator(central_longitude=0))
ax.coastlines()
ax.plot(groundings_arvor['lon'],groundings_arvor['lat'],'.',markersize=1,transform=ccrs.PlateCarree())

In [ ]:
#groundings_solo = pd.read_csv('/home1/datawork/kbalem/deep_argo_bathy_world_grnd.csv',header=None, names=['lat','lon','grnd','z'])
groundings_solo = pd.read_csv('Deep_SOLO_dos_paper3_10Jan2025.txt',delimiter=' ',names=['WMO', 'lon', 'lat', 'grnd', 'uncertainty', 'tid_flag', 'SID','gebco'])
groundings_solo.head()

In [ ]:
f=plt.figure()
ax=f.add_subplot(projection=ccrs.Mercator(central_longitude=180))
ax.coastlines()
ax.plot(groundings_solo['lon'],groundings_solo['lat'],'.',markersize=1,transform=ccrs.PlateCarree())

In [ ]:
# GEBCO BATHY GRID
mbgrid = xr.open_dataset('/home1/datawork/kbalem/GEBCO24/GEBCO_2024_sub_ice_topo.nc')
mbgrid

In [ ]:
# GEBCO TID GRID
tidgrid = xr.open_dataset('/home1/datawork/kbalem/GEBCO24/GEBCO_2024_TID.nc')
tidgrid

In [ ]:
# Glorys mean to calculate horizontal uncertainty distance
glomu = xr.open_dataset('/home1/datawork/kbalem/Glorys_mean/mercatorglorys12v1_gl12_mean_uo-ave2000-2020-ave-all-deps.nc')
glomv = xr.open_dataset('/home1/datawork/kbalem/Glorys_mean/mercatorglorys12v1_gl12_mean_vo-ave2000-2020-ave-all-deps.nc')
glomus = xr.open_dataset('/home1/datawork/kbalem/Glorys_mean/mercatorglorys12v1_gl12_mean_uo-ave2000-2020-surf.nc')
glomvs = xr.open_dataset('/home1/datawork/kbalem/Glorys_mean/mercatorglorys12v1_gl12_mean_vo-ave2000-2020-surf.nc')
vspeed = 0.09 #m/s

In [ ]:
i=0
lat, lon, grnd = groundings_arvor['lat'].values[i], groundings_arvor['lon'].values[i], groundings_arvor['grnd'].values[i]
vduration = grnd / vspeed
du = vduration*glomu['uo'].interp(latitude=lat,longitude=lon).values
dv = vduration*glomv['vo'].interp(latitude=lat,longitude=lon).values
hdist = np.round(np.sqrt(du**2 + dv**2),1)
hdist

## Any direct measurement tids within 5km around each point ?

In [ ]:
def reduce_to_dist(lat,lon,d):
    r = np.ceil(d/110.)/2.
    subtgrid = tidgrid.sel(lon=slice(lon-r,lon+r),lat=slice(lat-r,lat+r))
    stacked_subtgrid = subtgrid.stack(k=('lat','lon'))
    stacked_subtgrid = stacked_subtgrid.dropna('k')
    around_points = np.column_stack([stacked_subtgrid['lat'].values,stacked_subtgrid['lon'].values])    
    result = haversine_distances(np.radians([lat,lon]).reshape(1, -1), np.radians(around_points))
    result=result*6371000/1000
    stacked_subtgrid['d']=('k',result.flatten())
    subtgrid = stacked_subtgrid.unstack()
    subtgrid = subtgrid.where(subtgrid['d']<=d,drop=True)
    return subtgrid

In [ ]:
tids={
0:'Land',
10:'Singlebeam',
11:'Multibeam',
12:'Seismic',
13:'Isolated sounding',
14:'ENC sounding',
15:'Lidar',
16:'Optical light sensor',
17:'Combination of direct measurement',
40:'Satellite-derived gravity data',
41:'Interpolated based on an algorithm',
42:'Digital bathymetric from charts',
43:'Digital bathymetric from ENCs',
44:'Bathymetric sounding',
45:'Predicted based on helicopter',
46:'Draft of a grounded iceberg',
70:'Pre-generated grid',
71:'Unknown source',
72:'Steering points'
}

In [ ]:
i=19
ds=groundings_arvor
print(ds['WMO'].values[i])
lat, lon, grnd = ds['lat'].values[i], ds['lon'].values[i], ds['grnd'].values[i]
# GEBCO
elev = mbgrid['elevation'].interp(lon=lon,lat=lat,method='nearest').values
# UNCERTAINTY
vduration = grnd / vspeed
du = vduration*glomu['uo'].interp(latitude=lat,longitude=lon).values
dv = vduration*glomv['vo'].interp(latitude=lat,longitude=lon).values
hdist = np.round(np.sqrt(du**2 + dv**2),1)
# TIDs
subtgrid = reduce_to_dist(lat,lon,10)
# PLOT
c = subtgrid['tid'].stack(k=("lat", "lon"))
uc = np.unique(c.values)
uc = uc[~np.isnan(uc)]
sc = [str(int(i))+' : '+tids[i] for i in uc]
cc = np.zeros_like(c.values)*np.nan
for i in range(len(c.values)):
    if(~np.isnan(c.values[i])):
        cc[i] = np.argwhere(c.values[i]==uc)[0].flatten()
c.values = cc
c = c.unstack()
f=plt.figure(figsize=(16,8))
ax=f.add_subplot(121,projection=ccrs.Mercator())
ax.gridlines(linestyle=':',draw_labels=True)
a=c.plot(cmap=plt.get_cmap('tab10',len(uc)),vmin=0,vmax=len(uc),add_colorbar=False,ax=ax,transform=ccrs.PlateCarree())
cbar = plt.colorbar(a,pad=0.2,shrink=0.6)
cbar.set_ticks(np.arange(len(uc))+0.5+.5,labels=sc,rotation=45,ha='center')
ax.plot(lon,lat,'r.',transform=ccrs.PlateCarree())
ax.text(lon,lat+.005,str(elev)+' m - ('+str(hdist)+' m)',color='k',transform=ccrs.PlateCarree())

ci = Geodesic.circle(lon, lat, 5000,100,endpoint=True)
ci_u = Geodesic.circle(lon, lat, hdist,100,endpoint=True)
ax.plot(ci[:,0],ci[:,1],'r',transform=ccrs.PlateCarree())
ax.plot(ci_u[:,0],ci_u[:,1],'g',transform=ccrs.PlateCarree())

ax1=f.add_subplot(122,projection=ccrs.Mercator())
ax1.gridlines(linestyle=':',draw_labels=True)
ax1.coastlines()
ax1.plot(lon,lat,'ro',transform=ccrs.PlateCarree())
ax1.set_global()

direct_tids = [10,11,12,13,14,15,16,17]
print("Any direct sounding within 5k ? : ",np.any([tid in subtgrid['tid'] for tid in direct_tids]))

In [ ]:
flagz_arvor = np.zeros(len(groundings_arvor),dtype=int)-1
elevz_arvor = np.zeros(len(groundings_arvor),dtype=float)
hdist_arvor = np.zeros(len(groundings_arvor),dtype=float)

direct_tids = [10,11,12,13,14,15,16,17]
radiuskm = 5

for i in range(len(flagz_arvor)):       
    lat, lon, grnd = groundings_arvor['lat'].values[i], groundings_arvor['lon'].values[i], groundings_arvor['grnd'].values[i]
    # GEBCO
    elevz_arvor[i] = mbgrid['elevation'].interp(lon=lon,lat=lat,method='nearest').values
    # UNCERTAINTY
    vduration = grnd / vspeed
    du = vduration*glomu['uo'].interp(latitude=lat,longitude=lon).values
    dv = vduration*glomv['vo'].interp(latitude=lat,longitude=lon).values
    hdist_arvor[i] = np.round(np.sqrt(du**2 + dv**2),1)
    # TID
    subtgrid = reduce_to_dist(lat,lon,radiuskm)
    if np.any([tid in subtgrid['tid'] for tid in direct_tids]):
        flagz_arvor[i]=9998    

In [ ]:
print('9998 : ',len(flagz_arvor[flagz_arvor==9998]))
print('-1 : ',len(flagz_arvor[flagz_arvor==-1]))

In [ ]:
f=plt.figure(figsize=(15,10))
ax=f.add_subplot(projection=ccrs.Mercator(central_longitude=0))
ax.coastlines()
a=ax.scatter(groundings_arvor['lon'],groundings_arvor['lat'],s=4,c=flagz_arvor,transform=ccrs.PlateCarree())

In [ ]:
groundings_arvor['uncertainty2'] = hdist_arvor
groundings_arvor['tid_flag2'] = flagz_arvor
groundings_arvor['gebco2'] = elevz_arvor

In [ ]:
groundings_arvor.to_csv('grounding_arvor_cmfile_25.csv',index=False)

In [ ]:
flagz_solo = np.zeros(len(groundings_solo),dtype=int)-1
elevz_solo = np.zeros(len(groundings_solo),dtype=float)
hdist_solo = np.zeros(len(groundings_solo),dtype=float)

direct_tids = [10,11,12,13,14,15,16,17]
radiuskm = 5

for i in range(len(flagz_solo)):       
    lat, lon, grnd = groundings_solo['lat'].values[i], groundings_solo['lon'].values[i], -groundings_solo['grnd'].values[i]
    # GEBCO
    elevz_solo[i] = mbgrid['elevation'].interp(lon=lon,lat=lat,method='nearest').values
    # UNCERTAINTY
    vduration = grnd / vspeed
    du = vduration*glomu['uo'].interp(latitude=lat,longitude=lon).values
    dv = vduration*glomv['vo'].interp(latitude=lat,longitude=lon).values
    hdist_solo[i] = np.round(np.sqrt(du**2 + dv**2),1)
    # TID
    subtgrid = reduce_to_dist(lat,lon,radiuskm)
    if np.any([tid in subtgrid['tid'] for tid in direct_tids]):
        flagz_solo[i]=9998    

In [ ]:
print('9998 : ',len(flagz_solo[flagz_solo==9998]))
print('-1 : ',len(flagz_solo[flagz_solo==-1]))

In [ ]:
f=plt.figure(figsize=(15,10))
ax=f.add_subplot(projection=ccrs.Mercator(central_longitude=0))
ax.coastlines()
a=ax.scatter(groundings_solo['lon'],groundings_solo['lat'],s=4,c=flagz_solo,transform=ccrs.PlateCarree())

In [ ]:
groundings_solo['uncertainty2'] = hdist_solo
groundings_solo['tid_flag2'] = flagz_solo
groundings_solo['gebco2'] = elevz_solo

In [ ]:
groundings_solo.to_csv('grounding_solo_cmfile_25.csv',index=False)